In [5]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Loading
df = pd.read_csv('cleaned_drug_side_effects.csv')

# Utility function to transform strings "A;B;C" into lists ['A', 'B', 'C']
def split_items(x):
   return x.split(';') if pd.notnull(x) else []

# --- A. Encoding Targets (Explanatory Variables) ---
# Transform the column into a list
df['targets_list'] = df['targets'].apply(split_items)

# MultiLabelBinarizer creates one column per possible target
mlb_targets = MultiLabelBinarizer()
X_targets = mlb_targets.fit_transform(df['targets_list'])
print(f"Shape of 'targets' features: {X_targets.shape}") 
# Expected result: (1052, ~1148)

Shape of 'targets' features: (1052, 1400)


# Encoding Names (Explanatory Variables)
We use TF-IDF on characters (n-grams of size 3 to 5).

Capturing roots like "meth", "oxy", "zole".
analyzer='char': Instead of reading entire words (like "cat", "dog"), the algorithm will read characters (letters).

This is crucial because drug names do not have "sentences", the meaning is hidden within the word itself. ngram_range=(3, 5): This instructs the algorithm to create a sliding window that captures groups of 3, 4, and 5 consecutive letters.

In [7]:
# --- B. Encoding Names (Explanatory Variables) ---

tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))

# Fit: It scans through the entire drug_name column, splits all names into n-grams (3 to 5 letters) and builds a gigantic dictionary of all existing combinations (e.g., azep, zepa, epam...).
# Transform: It replaces each drug name with a numerical vector (a sequence of numbers).
X_names = tfidf.fit_transform(df['drug_name'])

print(f"Shape of 'names' features: {X_names.shape}")

Shape of 'names' features: (1052, 10205)


In [8]:
# --- C. Encoding of Side Effects (Target to Predict) ---
# Transform the column into a list
df['side_effects_list'] = df['side_effect'].apply(split_items)

# MultiLabelBinarizer creates one column per possible side effect
mlb_effects = MultiLabelBinarizer()
y = mlb_effects.fit_transform(df['side_effects_list'])

# Print the shape of the target variable
print(f"Shape of the target 'y': {y.shape}")
# Expected result: (1052, ~5735)

Shape of the target 'y': (1052, 5735)


In [10]:


# --- D. Fusion for the model ---
# To obtain your final matrix X, concatenate the features
X_final = np.hstack([X_targets, X_names.toarray()])

print(f"Final matrix for training: {X_final.shape}")
print(f"First rows of the final matrix X_final:\n{X_final[:5]}")  # Displays the first 5 rows and 5 columns

Final matrix for training: (1052, 11605)
First rows of the final matrix X_final:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
